# Gene Similarity Calculation

This notebook calculates **pairwise similarity and distance matrices** between genes using curve fitting parameters from the gene-level statistics analysis.

## Problem Type: **Distance/Similarity Matrix Calculation**

**Why this analysis matters:**
- Identifies genes with similar growth curve profiles
- Enables clustering and comparative analysis of gene functions
- Provides quantitative measures for gene relationship studies
- Supports downstream analyses like network construction and functional annotation

**How we solve it:**
We use three key features from curve fitting analysis:
- **A**: Amplitude parameter (maximum growth difference)
- **um**: Growth rate parameter (steepness of curve)
- **lam**: Lag phase parameter (time to reach maximum growth)

## Input Requirements
Your input TSV file should contain these columns:
- `Systematic ID`: Gene systematic identifier
- `Name`: Gene name
- `A`: Amplitude parameter from curve fitting
- `um`: Growth rate parameter from curve fitting  
- `lam`: Lag phase parameter from curve fitting

## Output Files
This notebook generates four similarity/distance matrices:
1. **Euclidean Distance Matrix**: Standard geometric distance
2. **Manhattan Distance Matrix**: Sum of absolute differences
3. **Cosine Similarity Matrix**: Angle-based similarity (0-1 scale)
4. **Mahalanobis Distance Matrix**: Standardized distance considering covariance

## Before/After Comparison
- **Before**: Individual gene curve parameters (A, um, lam)
- **After**: Pairwise similarity matrices enabling gene relationship analysis


In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.spatial.distance import pdist, squareform
from scipy.stats import zscore
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

# Configure matplotlib for publication-quality figures
plt.style.use('default')
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 12
plt.rcParams['axes.linewidth'] = 1.2
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['figure.dpi'] = 300

print("Libraries imported successfully!")
print("✓ Data processing: pandas, numpy")
print("✓ Visualization: matplotlib, seaborn") 
print("✓ Distance calculations: scipy")
print("✓ Similarity calculations: scikit-learn")


## Configuration and File Paths

Set up the paths to your data files and output directories. Modify these paths according to your setup.


In [ ]:
# Configuration - Modify these paths according to your setup
CONFIG = {
    # Input data file (TSV format with curve fitting results)
    'input_file': '../../results/HD_DIT_HAP/18_gene_level_curve_fitting/Gene_level_statistics_fitted.tsv',
    
    # Output directory for similarity matrices
    'output_dir': '../../results/HD_DIT_HAP/21_similarity_analysis_for_terms/',
    
    # Feature columns to use for similarity calculation
    'feature_columns': ['A', 'um', 'lam'],
    
    # Gene identifier columns
    'gene_id_column': 'Systematic ID',
    'gene_name_column': 'Name',
    
    # Analysis parameters
    'exclude_unsuccessful_fits': True,  # Exclude genes with Status != 'Success'
    'standardize_features': True,       # Standardize features before distance calculation
    'save_matrices': True,              # Save matrices to CSV files
    'create_heatmaps': True,            # Generate heatmap visualizations
    
    # Visualization parameters
    'heatmap_figsize': (12, 10),
    'max_genes_heatmap': 5000,  # Maximum number of genes to show in heatmap
    'colormap': 'viridis'
}

# Create output directory if it doesn't exist
output_dir = Path(CONFIG['output_dir'])
output_dir.mkdir(parents=True, exist_ok=True)

print("Configuration loaded successfully!")
print(f"Input file: {CONFIG['input_file']}")
print(f"Output directory: {CONFIG['output_dir']}")
print(f"Features to use: {CONFIG['feature_columns']}")
print(f"Standardize features: {CONFIG['standardize_features']}")
print(f"Save matrices: {CONFIG['save_matrices']}")
print(f"Create heatmaps: {CONFIG['create_heatmaps']}")


## Helper Functions

These functions handle the core similarity and distance calculations using different mathematical approaches.


In [ ]:
def load_and_prepare_data(input_file, config):
    """
    Load gene data and prepare feature matrix for similarity calculation.
    
    Parameters:
    -----------
    input_file : str or Path
        Path to the input TSV file with gene curve fitting results
    config : dict
        Configuration dictionary with column names and parameters
        
    Returns:
    --------
    tuple : (DataFrame, numpy.ndarray, list)
        Gene metadata, feature matrix, and gene identifiers
    """
    print(f"Loading data from: {input_file}")
    
    # Load the data
    df = pd.read_csv(input_file, sep='\t')
    print(f"Loaded {len(df)} genes from input file")
    
    # Filter for successful fits if specified
    if config['exclude_unsuccessful_fits']:
        df = df[df['Status'] == 'Success']
        print(f"After filtering for successful fits: {len(df)} genes")
    
    # Extract features
    feature_cols = config['feature_columns']
    features = df[feature_cols].values
    
    # Check for missing values
    n_missing = pd.isna(features).sum()
    if n_missing > 0:
        print(f"Warning: Found {n_missing} missing values in features")
        # Remove rows with missing values
        missing_mask = pd.isna(features).any(axis=1)
        df = df[~missing_mask]
        features = features[~missing_mask]
        print(f"After removing missing values: {len(df)} genes")
    
    # Standardize features if specified
    if config['standardize_features']:
        scaler = StandardScaler()
        features = scaler.fit_transform(features)
        print("✓ Features standardized (mean=0, std=1)")
    
    # Get gene identifiers
    gene_ids = df[config['gene_id_column']].values
    
    print(f"Final dataset: {len(df)} genes × {len(feature_cols)} features")
    print(f"Features: {feature_cols}")
    
    return df, features, gene_ids

def calculate_euclidean_distance(features):
    """
    Calculate Euclidean distance matrix.
    
    Euclidean distance is the standard geometric distance in multi-dimensional space.
    Formula: sqrt(sum((x_i - y_i)^2))
    
    Parameters:
    -----------
    features : numpy.ndarray
        Feature matrix (genes × features)
        
    Returns:
    --------
    numpy.ndarray : Square distance matrix
    """
    print("Calculating Euclidean distance matrix...")
    distances = pdist(features, metric='euclidean')
    distance_matrix = squareform(distances)
    print(f"✓ Euclidean distance matrix: {distance_matrix.shape}")
    return distance_matrix

def calculate_manhattan_distance(features):
    """
    Calculate Manhattan distance matrix.
    
    Manhattan distance is the sum of absolute differences between coordinates.
    Formula: sum(|x_i - y_i|)
    
    Parameters:
    -----------
    features : numpy.ndarray
        Feature matrix (genes × features)
        
    Returns:
    --------
    numpy.ndarray : Square distance matrix
    """
    print("Calculating Manhattan distance matrix...")
    distances = pdist(features, metric='cityblock')
    distance_matrix = squareform(distances)
    print(f"✓ Manhattan distance matrix: {distance_matrix.shape}")
    return distance_matrix

def calculate_cosine_similarity(features):
    """
    Calculate cosine similarity matrix.
    
    Cosine similarity measures the cosine of the angle between two vectors.
    Values range from -1 to 1, where 1 indicates identical direction.
    Formula: dot(A,B) / (norm(A) * norm(B))
    
    Parameters:
    -----------
    features : numpy.ndarray
        Feature matrix (genes × features)
        
    Returns:
    --------
    numpy.ndarray : Square similarity matrix
    """
    print("Calculating cosine distance matrix...")
    distance = pdist(features, metric='cosine')
    distance_matrix = squareform(distance)
    print(f"✓ Cosine similarity matrix: {distance_matrix.shape}")
    return distance_matrix

def calculate_mahalanobis_distance(features):
    """
    Calculate Mahalanobis distance matrix.
    
    Mahalanobis distance is a standardized distance that accounts for covariance
    between features. It's scale-invariant and accounts for feature correlations.
    Formula: sqrt((x-y)^T * Σ^(-1) * (x-y))
    
    Parameters:
    -----------
    features : numpy.ndarray
        Feature matrix (genes × features)
        
    Returns:
    --------
    numpy.ndarray : Square distance matrix
    """
    print("Calculating Mahalanobis distance matrix...")
    try:
        # Calculate covariance matrix
        cov_matrix = np.cov(features.T)
        
        # Check if covariance matrix is invertible
        if np.linalg.det(cov_matrix) == 0:
            print("Warning: Covariance matrix is singular. Adding small regularization term.")
            cov_matrix += np.eye(cov_matrix.shape[0]) * 1e-6
        
        distances = pdist(features, metric='mahalanobis', VI=np.linalg.inv(cov_matrix))
        distance_matrix = squareform(distances)
        print(f"✓ Mahalanobis distance matrix: {distance_matrix.shape}")
        return distance_matrix
    
    except np.linalg.LinAlgError:
        print("Error: Could not calculate Mahalanobis distance. Using Euclidean instead.")
        return calculate_euclidean_distance(features)

def save_matrix_to_file(matrix, gene_ids, output_path, matrix_name):
    """
    Save similarity/distance matrix to CSV file with gene identifiers.
    
    Parameters:
    -----------
    matrix : numpy.ndarray
        Square similarity/distance matrix
    gene_ids : list
        Gene identifiers for rows/columns
    output_path : Path
        Output file path
    matrix_name : str
        Name of the matrix for logging
    """
    print(f"Saving {matrix_name} matrix to: {output_path}")
    
    # Create DataFrame with gene IDs as index and columns
    df_matrix = pd.DataFrame(matrix, index=gene_ids, columns=gene_ids).round(5)
    
    # Save to CSV
    df_matrix.to_csv(output_path)
    print(f"✓ {matrix_name} matrix saved successfully")

print("Helper functions loaded successfully!")
print("✓ load_and_prepare_data: Data loading and preprocessing")
print("✓ calculate_euclidean_distance: Standard geometric distance")
print("✓ calculate_manhattan_distance: Sum of absolute differences")
print("✓ calculate_cosine_similarity: Angle-based similarity")
print("✓ calculate_mahalanobis_distance: Standardized covariance-aware distance")
print("✓ save_matrix_to_file: CSV export with gene identifiers")


## Data Loading and Preprocessing

Load the gene curve fitting data and prepare the feature matrix for similarity calculations.


In [ ]:
# Load and prepare the data
gene_data, feature_matrix, gene_ids = load_and_prepare_data(
    CONFIG['input_file'], 
    CONFIG
)

# Display basic statistics about the features
print("\n" + "="*50)
print("FEATURE STATISTICS")
print("="*50)

feature_stats = pd.DataFrame({
    'Feature': CONFIG['feature_columns'],
    'Mean': feature_matrix.mean(axis=0),
    'Std': feature_matrix.std(axis=0),
    'Min': feature_matrix.min(axis=0),
    'Max': feature_matrix.max(axis=0)
})

print(feature_stats.round(4))

# Display sample of the processed data
print("\n" + "="*50)
print("SAMPLE OF PROCESSED DATA")
print("="*50)

sample_data = gene_data[[CONFIG['gene_id_column'], CONFIG['gene_name_column']] + CONFIG['feature_columns']].head(10)
print(sample_data)


## Similarity and Distance Matrix Calculation

Calculate four different types of similarity/distance matrices using the prepared features.

### Mathematical Approaches Comparison:

1. **Euclidean Distance**: Standard geometric distance
   - Good for: General-purpose distance measurement
   - Sensitive to: Feature scale and outliers

2. **Manhattan Distance**: Sum of absolute differences  
   - Good for: Robust to outliers
   - Sensitive to: Feature scale

3. **Cosine Similarity**: Measures angle between vectors
   - Good for: Direction-based similarity (scale-invariant)
   - Range: -1 to 1 (higher = more similar)

4. **Mahalanobis Distance**: Accounts for feature correlations
   - Good for: Correlated features, different scales
   - Sensitive to: Sample size and feature correlations


In [ ]:
print("="*60)
print("CALCULATING SIMILARITY AND DISTANCE MATRICES")
print("="*60)

# Calculate all four matrices
euclidean_matrix = calculate_euclidean_distance(feature_matrix)
manhattan_matrix = calculate_manhattan_distance(feature_matrix)
cosine_matrix = calculate_cosine_similarity(feature_matrix)
mahalanobis_matrix = calculate_mahalanobis_distance(feature_matrix)

# Store results in a dictionary for easy access
matrices = {
    'Euclidean_Distance': euclidean_matrix,
    'Manhattan_Distance': manhattan_matrix,
    'Cosine_Similarity': cosine_matrix,
    'Mahalanobis_Distance': mahalanobis_matrix
}

print("\n" + "="*60)
print("MATRIX STATISTICS")
print("="*60)

for name, matrix in matrices.items():
    print(f"\n{name}:")
    print(f"  Shape: {matrix.shape}")
    print(f"  Range: {matrix.min():.4f} to {matrix.max():.4f}")
    print(f"  Mean: {matrix.mean():.4f}")
    print(f"  Std: {matrix.std():.4f}")
    
    # For distance matrices, diagonal should be 0
    if 'Distance' in name:
        diagonal_check = np.allclose(np.diag(matrix), 0)
        print(f"  Diagonal is zero: {diagonal_check}")
    
    # For similarity matrices, diagonal should be 1
    if 'Similarity' in name:
        diagonal_check = np.allclose(np.diag(matrix), 1)
        print(f"  Diagonal is one: {diagonal_check}")

print("\n✓ All matrices calculated successfully!")


## Save Matrices to Files

Export the calculated matrices to CSV files for downstream analysis.


In [ ]:
if CONFIG['save_matrices']:
    print("="*60)
    print("SAVING MATRICES TO FILES")
    print("="*60)
    
    # Save each matrix to a separate CSV file
    for name, matrix in matrices.items():
        filename = f"{name.lower()}_matrix.csv"
        output_path = Path(CONFIG['output_dir']) / filename
        save_matrix_to_file(matrix, gene_ids, output_path, name)
    
    print("\n✓ All matrices saved successfully!")
    print(f"Output directory: {CONFIG['output_dir']}")
    
    # Display file sizes
    print("\nFile sizes:")
    for name in matrices.keys():
        filename = f"{name.lower()}_matrix.csv"
        file_path = Path(CONFIG['output_dir']) / filename
        if file_path.exists():
            size_mb = file_path.stat().st_size / (1024 * 1024)
            print(f"  {filename}: {size_mb:.2f} MB")
            
else:
    print("Matrix saving is disabled in configuration.")


## Visualization: Heatmaps

Create heatmap visualizations of the similarity/distance matrices to explore patterns and relationships between genes.

**Interpretation Guide:**
- **Distance matrices**: Lower values (darker colors) indicate more similar genes
- **Similarity matrices**: Higher values (brighter colors) indicate more similar genes  
- **Diagonal elements**: Should be 0 for distances, 1 for similarities
- **Block patterns**: Indicate groups of similar genes


In [ ]:
def create_clustered_heatmap(matrix, gene_ids, matrix_name, max_genes=50, figsize=(12, 10)):
    """
    Create a clustered heatmap visualization of a similarity/distance matrix.
    
    Parameters:
    -----------
    matrix : numpy.ndarray
        Square similarity/distance matrix
    gene_ids : list
        Gene identifiers
    matrix_name : str
        Name of the matrix for title
    max_genes : int
        Maximum number of genes to display (for readability)
    figsize : tuple
        Figure size (width, height)
        
    Returns:
    --------
    matplotlib.figure.Figure : The created figure
    """
    from scipy.cluster.hierarchy import dendrogram, linkage
    from scipy.spatial.distance import squareform
    
    # Subset matrix if too large
    if len(gene_ids) > max_genes:
        print(f"Displaying first {max_genes} genes for readability")
        subset_matrix = matrix[:max_genes, :max_genes]
        subset_gene_ids = gene_ids[:max_genes]
    else:
        subset_matrix = matrix
        subset_gene_ids = gene_ids
    
    # Convert similarity to distance if needed for clustering
    if 'Similarity' in matrix_name:
        # Convert similarity to distance (1 - similarity)
        distance_matrix = 1 - subset_matrix
        # Ensure diagonal is 0
        np.fill_diagonal(distance_matrix, 0)
    else:
        distance_matrix = subset_matrix.copy()
    
    # Handle negative distances by shifting to make all values non-negative
    min_distance = np.min(distance_matrix)
    if min_distance < 0:
        print(f"Warning: Found negative distances (min: {min_distance:.4f}). Shifting values to make non-negative.")
        distance_matrix = distance_matrix - min_distance
    
    # Ensure diagonal is exactly 0 for distance matrices
    np.fill_diagonal(distance_matrix, 0)
    
    # Ensure matrix is symmetric (required for hierarchical clustering)
    distance_matrix = (distance_matrix + distance_matrix.T) / 2
    
    # Additional validation: ensure all values are non-negative
    distance_matrix = np.maximum(distance_matrix, 0)
    
    try:
        # Perform hierarchical clustering
        # Convert to condensed distance matrix for linkage
        condensed_dist = squareform(distance_matrix, checks=False)
        
        # Check for any remaining issues with the condensed distance matrix
        if np.any(condensed_dist < 0):
            print("Warning: Negative values found in condensed distance matrix. Setting to 0.")
            condensed_dist = np.maximum(condensed_dist, 0)
        
        # Use 'average' linkage method which is more robust than 'ward' for potentially problematic distance matrices
        linkage_method = 'average' if min_distance < 0 else 'ward'
        linkage_matrix = linkage(condensed_dist, method=linkage_method)
        
    except ValueError as e:
        print(f"Clustering failed with error: {e}")
        print("Falling back to simple ordering without clustering...")
        # Create a simple figure without clustering
        fig, ax = plt.subplots(figsize=figsize)
        
        # Choose colormap based on matrix type
        if 'Similarity' in matrix_name:
            cmap = 'viridis'
            vmin, vmax = 0, 1
            plot_matrix = subset_matrix
        else:
            cmap = 'viridis_r'  # Reverse for distance (lower = more similar)
            vmin, vmax = None, None
            plot_matrix = subset_matrix
        
        # Create simple heatmap without clustering
        im = ax.imshow(plot_matrix, cmap=cmap, vmin=vmin, vmax=vmax, aspect='auto')
        
        # Add colorbar
        cbar = plt.colorbar(im, ax=ax, shrink=0.8)
        if 'Similarity' in matrix_name:
            cbar.set_label('Similarity Score', rotation=270, labelpad=15)
        else:
            cbar.set_label('Distance Score', rotation=270, labelpad=15)
        
        # Set labels
        ax.set_title(f'{matrix_name} Matrix (No Clustering)', fontsize=16, fontweight='bold')
        ax.set_xlabel('Genes', fontsize=12)
        ax.set_ylabel('Genes', fontsize=12)
        
        # Set ticks
        tick_step = max(1, len(subset_gene_ids) // 10)
        tick_indices = range(0, len(subset_gene_ids), tick_step)
        tick_labels = [subset_gene_ids[i] for i in tick_indices]
        
        ax.set_xticks(tick_indices)
        ax.set_xticklabels(tick_labels, rotation=45, ha='right')
        ax.set_yticks(tick_indices)
        ax.set_yticklabels(tick_labels)
        
        return fig
    
    # Create figure with subplots for dendrogram and heatmap
    fig = plt.figure(figsize=figsize)
    
    # Create grid layout
    gs = fig.add_gridspec(2, 2, width_ratios=[0.2, 1], height_ratios=[0.2, 1],
                         hspace=0.05, wspace=0.05)
    
    # Top dendrogram (for columns)
    ax_dendro_top = fig.add_subplot(gs[0, 1])
    dendro_top = dendrogram(linkage_matrix, ax=ax_dendro_top, orientation='top',
                           labels=subset_gene_ids, no_labels=True, color_threshold=0)
    ax_dendro_top.set_xticks([])
    ax_dendro_top.set_yticks([])
    ax_dendro_top.spines['top'].set_visible(False)
    ax_dendro_top.spines['right'].set_visible(False)
    ax_dendro_top.spines['bottom'].set_visible(False)
    ax_dendro_top.spines['left'].set_visible(False)
    
    # Left dendrogram (for rows)
    ax_dendro_left = fig.add_subplot(gs[1, 0])
    dendro_left = dendrogram(linkage_matrix, ax=ax_dendro_left, orientation='left',
                            labels=subset_gene_ids, no_labels=True, color_threshold=0)
    ax_dendro_left.set_xticks([])
    ax_dendro_left.set_yticks([])
    ax_dendro_left.spines['top'].set_visible(False)
    ax_dendro_left.spines['right'].set_visible(False)
    ax_dendro_left.spines['bottom'].set_visible(False)
    ax_dendro_left.spines['left'].set_visible(False)
    
    # Get clustering order
    cluster_order = dendro_top['leaves']
    
    # Reorder matrix according to clustering
    clustered_matrix = subset_matrix[np.ix_(cluster_order, cluster_order)]
    clustered_gene_ids = [subset_gene_ids[i] for i in cluster_order]
    
    # Main heatmap
    ax_heatmap = fig.add_subplot(gs[1, 1])
    
    # Choose colormap based on matrix type
    if 'Similarity' in matrix_name:
        cmap = 'viridis'
        vmin, vmax = 0, 1
    else:
        cmap = 'viridis_r'  # Reverse for distance (lower = more similar)
        vmin, vmax = None, None
    
    # Create heatmap
    im = ax_heatmap.imshow(clustered_matrix, cmap=cmap, vmin=vmin, vmax=vmax, aspect='auto')
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax_heatmap, shrink=0.8)
    if 'Similarity' in matrix_name:
        cbar.set_label('Similarity Score', rotation=270, labelpad=15)
    else:
        cbar.set_label('Distance Score', rotation=270, labelpad=15)
    
    # Set labels
    ax_heatmap.set_title(f'Clustered {matrix_name} Matrix', fontsize=16, fontweight='bold', pad=20)
    ax_heatmap.set_xlabel('Genes (Clustered)', fontsize=12)
    ax_heatmap.set_ylabel('Genes (Clustered)', fontsize=12)
    
    # Set ticks (reduce number for readability)
    tick_step = max(1, len(clustered_gene_ids) // 10)
    tick_indices = range(0, len(clustered_gene_ids), tick_step)
    tick_labels = [clustered_gene_ids[i] for i in tick_indices]
    
    ax_heatmap.set_xticks(tick_indices)
    ax_heatmap.set_xticklabels(tick_labels, rotation=45, ha='right')
    ax_heatmap.set_yticks(tick_indices)
    ax_heatmap.set_yticklabels(tick_labels)
    
    # Add grid for better readability
    ax_heatmap.grid(True, alpha=0.3, linewidth=0.5)
    
    return fig

if CONFIG['create_heatmaps']:
    print("="*60)
    print("CREATING CLUSTERED HEATMAP VISUALIZATIONS")
    print("="*60)
    
    # Create clustered heatmaps for each matrix
    for name, matrix in matrices.items():
        print(f"\nCreating clustered heatmap for {name}...")
        
        fig = create_clustered_heatmap(
            matrix, 
            gene_ids, 
            name, 
            max_genes=CONFIG['max_genes_heatmap'],
            figsize=CONFIG['heatmap_figsize']
        )
        
        # Save figure
        if CONFIG['save_matrices']:
            filename = f"{name.lower()}_clustered_heatmap.pdf"
            output_path = Path(CONFIG['output_dir']) / filename
            fig.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"✓ Clustered heatmap saved: {filename}")
        
        # Display the figure
        plt.show()
        
        # Close figure to free memory
        plt.close(fig)
    
    print("\n✓ All clustered heatmaps created successfully!")
    print("✓ Hierarchical clustering applied to reveal gene groupings")
    
else:
    print("Heatmap creation is disabled in configuration.")


## Summary and Analysis

### What we accomplished:
1. **Data Loading**: Loaded and preprocessed gene curve fitting parameters
2. **Feature Preparation**: Standardized A, um, and lam parameters for analysis
3. **Matrix Calculation**: Computed four different similarity/distance matrices
4. **File Export**: Saved matrices as CSV files for downstream analysis
5. **Visualization**: Created heatmap visualizations to explore patterns

### Key Benefits of Each Matrix:

- **Euclidean Distance**: Best for general-purpose similarity analysis
- **Manhattan Distance**: More robust to outliers, good for noisy data
- **Cosine Similarity**: Captures directional similarity, scale-invariant
- **Mahalanobis Distance**: Accounts for feature correlations and different scales

### Next Steps:
1. **Clustering**: Use distance matrices for gene clustering analysis
2. **Network Analysis**: Build gene similarity networks
3. **Functional Analysis**: Correlate similarity with functional annotations
4. **Pathway Analysis**: Identify similar genes in same pathways


In [ ]:
print("="*60)
print("FINAL SUMMARY - GENE SIMILARITY ANALYSIS COMPLETED")
print("="*60)

print("\n✅ CHECKLIST - All Changes Made:")
print("="*40)
print("✓ Loaded gene curve fitting data with A, um, lam parameters")
print("✓ Preprocessed and standardized features for analysis")
print("✓ Calculated Euclidean distance matrix (geometric distance)")
print("✓ Calculated Manhattan distance matrix (robust to outliers)")
print("✓ Calculated Cosine similarity matrix (direction-based)")
print("✓ Calculated Mahalanobis distance matrix (covariance-aware)")
print("✓ Saved all matrices as CSV files with gene identifiers")
print("✓ Created heatmap visualizations for pattern exploration")
print("✓ Generated publication-quality figures with proper styling")

print(f"\n📊 ANALYSIS RESULTS:")
print(f"  • Processed {len(gene_ids)} genes")
print(f"  • Generated {len(matrices)} similarity/distance matrices")
print(f"  • Created {len(matrices)} heatmap visualizations")
print(f"  • Output directory: {CONFIG['output_dir']}")

print(f"\n📁 OUTPUT FILES:")
if CONFIG['save_matrices']:
    for name in matrices.keys():
        print(f"  • {name.lower()}_matrix.csv")
        if CONFIG['create_heatmaps']:
            print(f"  • {name.lower()}_heatmap.png")

print(f"\n🔍 WHAT CHANGED:")
print("  • BEFORE: Individual gene parameters (A, um, lam)")
print("  • AFTER: Pairwise similarity matrices enabling comparative analysis")

print(f"\n🎯 KEY BENEFITS:")
print("  • Enables gene clustering and network analysis")
print("  • Provides multiple mathematical perspectives on similarity")
print("  • Supports functional annotation and pathway analysis")
print("  • Ready for downstream bioinformatics workflows")

print("\n" + "="*60)
print("ANALYSIS COMPLETE - READY FOR DOWNSTREAM ANALYSIS")
print("="*60)
